In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
Id = test['Id']

In [ ]:
# let's just use some variables
cols = ['LotFrontage', 'LotArea', 'YrSold']

In [ ]:
train = train[cols + ['SalePrice']]
test = test[cols]

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

In [ ]:
train

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

# Prepare X and y
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

# Create a list of k values to test

#k_values = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43]
k_values = [i for i in range(3, 46, 2)]

# Dictionary to store cross-validation scores for each k
cv_scores = {}

# Create and evaluate pipeline for each k value
for k in k_values:
    # Create pipeline
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=k))
    ])
    
    # Perform 5-fold cross validation
    scores = cross_val_score(
        pipeline, 
        X, 
        y,
        cv=5,
        scoring='neg_mean_squared_log_error'  # Using RMSE as metric
    )
    
    # Store absolute value of scores (converting from negative to positive)
    cv_scores[k] = -scores.mean()



In [ ]:
#from sklearn.metrics import get_scorer_names
#get_scorer_names()

In [ ]:
# Find the best k
best_k = min(cv_scores, key=cv_scores.get)
print("\nCross-validation MSLE for different k values:")
for k, rmse in cv_scores.items():
    print(f"k={k}: MSLE={rmse:,.2f}")
print(f"\nBest k: {best_k} with MSLE={cv_scores[best_k]:,.2f}")

In [ ]:

# Create final pipeline with best k
final_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(n_neighbors=best_k))
])

# Fit the final pipeline
final_pipeline.fit(X, y)

# Make predictions on test set
predictions = final_pipeline.predict(test)

# Create submission DataFrame
submission = pd.DataFrame({
    'Id': Id,
    'SalePrice': predictions
})


In [ ]:
submission.to_csv('submission.csv', index = False)

In [ ]:
df = pd.DataFrame(list(cv_scores.items()), columns=['k', 'score'])

print(df)

In [ ]:
import seaborn as sns

In [ ]:
sns.scatterplot(data = df, x = "k", y = "score")